# FinFlow Automated Verification — Guided Pre/Post Analysis

This notebook demonstrates a structured Pre/Post impact analysis when **no randomized Control group exists**.

**Business decision:** Did FinFlow's automated verification workflow improve transaction-processing performance enough to continue scaling, while keeping payment quality, fraud, and support guardrails acceptable?

**Intervention date:** 2026-04-01  
**Primary KPI:** verification completion rate  
**Secondary KPIs:** verification time, manual review rate, completed transaction volume  
**Guardrails:** payment decline rate, support contact rate, fraud-confirmed rate

All data are synthetic and created for training and portfolio demonstration.

## Why this case is different from A/B testing

In an A/B test, randomization provides a contemporaneous Control group. Here, everyone receives the new workflow at launch. Therefore, **Post is not the same thing as Treatment** and **Pre is not a randomized Control**.

A simple statement such as *completion was higher after launch* is not enough to conclude that the launch caused the increase. Baseline trend, seasonality, traffic mix, campaigns, implementation ramp, and external conditions can also move the KPI.

This notebook builds evidence in layers: data quality → descriptive change → stable Post → trends → mix/confounders → Interrupted Time Series → guardrails → business recommendation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
module_root = None
for candidate in [cwd] + list(cwd.parents):
    if (candidate / 'src' / 'generate_synthetic_data.py').exists() and (candidate / 'src' / 'analyze_pre_post.py').exists():
        module_root = candidate
        break
    if (candidate / '02_pre_post_analysis' / 'src' / 'generate_synthetic_data.py').exists():
        module_root = candidate / '02_pre_post_analysis'
        break

if module_root is None:
    raise FileNotFoundError('Could not locate the 02_pre_post_analysis module root.')

sys.path.insert(0, str(module_root / 'src'))

from generate_synthetic_data import generate_raw_data
from analyze_pre_post import (
    LAUNCH_DATE, RAMP_END, CAMPAIGN_START, CAMPAIGN_END,
    quality_report, clean_data, kpi_summary, two_proportion_pre_post,
    continuous_pre_post, daily_aggregate, interrupted_time_series,
    mix_comparison, segment_completion, stable_post_mask
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
ALPHA = 0.05

## 1. Frame the business question

> **Did automated verification reduce processing friction and manual-review demand without increasing payment declines, fraud risk, or support demand?**

Expected positive signals are higher verification completion, lower verification time, lower manual review, and lower or stable support contacts. Payment decline and fraud are guardrails. Because there is no randomized Control, causal confidence must be earned rather than assumed.

## 2. Generate and validate the synthetic raw data

The generator intentionally includes duplicates, inconsistent category casing, missing context fields, anomalous verification durations, corrupted intervention flags, baseline trend, weekday/weekend seasonality, launch ramp, traffic-mix shift, and a temporary marketing campaign.

In [ ]:
raw = generate_raw_data()
print(f'Raw rows: {len(raw):,}')
display(raw.head())
display(pd.Series(quality_report(raw), name='count').to_frame())

### Cleaning principle: re-derive intervention timing

Do not trust a raw `post_flag` when the intervention date is known. The cleaning workflow derives Pre/Post, ramp, campaign, and days-from-launch from `transaction_date`, standardizes categories, removes duplicate transaction IDs, and excludes impossible duration values only from duration calculations.

In [ ]:
df = clean_data(raw)
print(f'Final analytical rows: {len(df):,}')
print('Date range:', df['transaction_date'].min().date(), 'to', df['transaction_date'].max().date())
print('Launch:', LAUNCH_DATE.date(), '| Ramp end:', RAMP_END.date())
print('Campaign:', CAMPAIGN_START.date(), 'to', CAMPAIGN_END.date())

## 3. Start with the raw Pre/Post KPI comparison

Descriptive Pre/Post results answer **what changed**, not yet **what caused the change**. Review the primary KPI, secondary outcomes, and guardrails together before formal inference.

In [ ]:
summary = kpi_summary(df)
display(summary)

rate_cols = ['verification_completion_rate','manual_review_rate','payment_decline_rate','support_contact_rate','fraud_confirmed_rate']
ax = (summary[rate_cols].T * 100).plot(kind='bar', figsize=(10,5))
ax.set_ylabel('Rate (%)')
ax.set_title('FinFlow — Pre vs Post KPI Summary')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

## 4. Primary KPI: simple Pre/Post test

Verification completion is binary, so a two-proportion comparison quantifies the observed lift and its statistical uncertainty. A small p-value does **not** prove the intervention caused the difference.

In [ ]:
primary = two_proportion_pre_post(df, 'verification_completed')
display(pd.Series(primary.__dict__).to_frame('value'))
print(f'Pre: {primary.pre_rate:.2%}')
print(f'Post: {primary.post_rate:.2%}')
print(f'Absolute change: {primary.absolute_change*100:.2f} pp')
print(f'Relative change: {primary.relative_change:.2%}')
print(f'95% CI: [{primary.ci_low*100:.2f}, {primary.ci_high*100:.2f}] pp')
print(f'p-value: {primary.p_value:.6g}')

## 5. Compare full Post with stable Post

The first seven days are an intentional launch-ramp period. A good analysis reports the ramp separately and checks whether performance strengthens after stabilization rather than silently removing inconvenient observations.

In [ ]:
stable = two_proportion_pre_post(df, 'verification_completed', stable_post_mask(df))
display(pd.DataFrame({
    'Full Post': {'Pre rate': primary.pre_rate, 'Post rate': primary.post_rate, 'Change (pp)': primary.absolute_change*100, 'p-value': primary.p_value},
    'Stable Post': {'Pre rate': stable.pre_rate, 'Post rate': stable.post_rate, 'Change (pp)': stable.absolute_change*100, 'p-value': stable.p_value}
}).T)

## 6. Inspect trends, seasonality, and concurrent events

Before fitting a model, visualize the daily KPI and volume. Look for baseline trend, an immediate launch shift, a post-launch slope change, first-week ramp behavior, campaign effects, and weekday/weekend patterns.

In [ ]:
daily = daily_aggregate(df)
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(daily['transaction_date'], daily['verification_completion_rate']*100)
ax.axvline(LAUNCH_DATE, linestyle='--', label='Launch')
ax.axvspan(LAUNCH_DATE, RAMP_END, alpha=.15, label='Ramp')
ax.axvspan(CAMPAIGN_START, CAMPAIGN_END, alpha=.10, label='Campaign')
ax.set_ylabel('Verification completion (%)')
ax.set_title('Daily Verification Completion Rate')
ax.legend(); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(11,5))
ax.plot(daily['transaction_date'], daily['transactions'])
ax.axvline(LAUNCH_DATE, linestyle='--', label='Launch')
ax.axvspan(CAMPAIGN_START, CAMPAIGN_END, alpha=.10, label='Campaign')
ax.set_ylabel('Eligible transactions')
ax.set_title('Daily Transaction Volume')
ax.legend(); plt.tight_layout(); plt.show()

## 7. Check traffic mix

If the type of traffic changes after launch, part of the KPI movement may reflect composition rather than the workflow. Compare country, device, tenure, and risk-tier distributions.

In [ ]:
for column in ['country','device_type','customer_tenure','risk_tier']:
    print(f'\n{column}')
    display(mix_comparison(df, column))

## 8. Verification time and guardrails

Verification time is right-skewed, so report both mean and median and use Welch's t-test plus a non-parametric sensitivity check. For rare fraud events, inspect absolute event counts as well as percentage changes.

In [ ]:
display(pd.Series(continuous_pre_post(df, 'verification_time_seconds')).to_frame('value'))

guardrail_rows = []
for metric in ['manual_review','payment_declined','support_contact','fraud_confirmed']:
    r = two_proportion_pre_post(df, metric)
    guardrail_rows.append({
        'metric': metric, 'pre_rate': r.pre_rate, 'post_rate': r.post_rate,
        'change_pp': r.absolute_change*100, 'p_value': r.p_value,
        'pre_events': int(df.loc[df['period'].eq('Pre'), metric].sum()),
        'post_events': int(df.loc[df['period'].eq('Post'), metric].sum())
    })
guardrails = pd.DataFrame(guardrail_rows).set_index('metric')
display(guardrails)

## 9. Interrupted Time Series (ITS)

ITS uses the sequence of observations over time rather than treating Pre and Post as two undifferentiated buckets. The model estimates a baseline trend, an immediate level change at launch, and a post-launch slope change. The adjusted model also includes ramp, campaign, weekday effects, and daily traffic-mix covariates.

This strengthens the design, but it does not turn observational data into a randomized experiment.

In [ ]:
basic_model, _ = interrupted_time_series(df, adjusted=False)
adjusted_model, adjusted_daily = interrupted_time_series(df, adjusted=True)

print('Basic ITS')
display(basic_model.summary2().tables[1].loc[['time_index','post','time_after_launch']])

print('Adjusted ITS')
terms = ['time_index','post','time_after_launch','ramp','campaign']
coef_table = adjusted_model.summary2().tables[1].loc[terms].copy()
coef_table['Coef._pp'] = coef_table['Coef.'] * 100
display(coef_table)

### Reading the key coefficients

- `time_index`: estimated baseline daily trend before launch
- `post`: estimated immediate level shift at launch
- `time_after_launch`: estimated change in slope after launch
- `ramp`: temporary first-week effect
- `campaign`: temporary campaign-period effect

The outcome is a daily rate, so multiplying a coefficient by 100 gives an approximate percentage-point interpretation.

## 10. Model-based counterfactual

A stakeholder-friendly ITS chart can compare actual daily performance with fitted performance and a no-intervention counterfactual. The counterfactual is a model estimate, not observed truth, so its credibility depends on the model and the quality of measured confounders.

In [ ]:
daily_model = adjusted_daily.copy()
daily_model['fitted_observed'] = adjusted_model.predict(daily_model)
counterfactual = daily_model.copy()
counterfactual['post'] = 0
counterfactual['time_after_launch'] = 0
counterfactual['ramp'] = 0
daily_model['counterfactual'] = adjusted_model.predict(counterfactual)

fig, ax = plt.subplots(figsize=(11,5))
ax.plot(daily_model['transaction_date'], daily_model['verification_completion_rate']*100, label='Actual', alpha=.65)
ax.plot(daily_model['transaction_date'], daily_model['fitted_observed']*100, label='Adjusted fitted')
ax.plot(daily_model['transaction_date'], daily_model['counterfactual']*100, linestyle='--', label='No-intervention counterfactual')
ax.axvline(LAUNCH_DATE, linestyle=':', label='Launch')
ax.set_ylabel('Verification completion (%)')
ax.set_title('Interrupted Time Series — Observed vs Model Counterfactual')
ax.legend(); plt.tight_layout(); plt.show()

## 11. Segment analysis and business impact

Segments explain where the observed change is stronger or weaker, but in a Pre/Post design they may reflect both true heterogeneity and composition changes. For business impact, use the most defensible effect estimate rather than automatically choosing the largest one.

In [ ]:
for segment in ['device_type','country','customer_tenure','risk_tier']:
    print(f'\n{segment}')
    display(segment_completion(df, segment))

ANNUAL_ELIGIBLE_TRANSACTIONS = 3_000_000
adjusted_level_effect = float(adjusted_model.params['post'])
display(pd.Series({
    'Annual eligible transactions': ANNUAL_ELIGIBLE_TRANSACTIONS,
    'Observed simple Pre/Post lift (pp)': primary.absolute_change*100,
    'Observed implied incremental completions': ANNUAL_ELIGIBLE_TRANSACTIONS*primary.absolute_change,
    'Adjusted ITS immediate level effect (pp)': adjusted_level_effect*100,
    'Adjusted ITS implied incremental completions': ANNUAL_ELIGIBLE_TRANSACTIONS*adjusted_level_effect
}))

## 12. Rate causal confidence before recommending action

Evidence supporting the intervention explanation includes timing around launch, direction consistent with the expected mechanism, stronger stable-Post performance, a positive adjusted ITS effect, supportive operational outcomes, and acceptable guardrails.

Evidence limiting causal confidence includes the lack of randomization, baseline trend, traffic-mix changes, the marketing campaign, and possible unmeasured concurrent changes.

A defensible conclusion may therefore be: **the evidence is consistent with the automated workflow improving performance, and the adjusted time-series analysis strengthens that interpretation; however, causal confidence remains below that of a randomized experiment.**

## 13. Recommendation framework

- **Continue / Scale** — meaningful improvement, supportive time-series evidence, acceptable guardrails.
- **Continue with monitoring** — positive result but moderate causal confidence or a rare guardrail needs more observation.
- **Iterate** — positive direction but ramp or segment findings reveal correctable friction.
- **Validate further** — attribution remains too uncertain.
- **Roll back / Stop** — meaningful guardrail harm or insufficient value.

In [ ]:
primary_positive = primary.absolute_change > 0 and primary.p_value < ALPHA
its_positive = adjusted_model.params.get('post', 0) > 0 and adjusted_model.pvalues.get('post', 1) < ALPHA
adverse_guardrail = (
    (guardrails.loc['payment_declined','change_pp'] > 0 and guardrails.loc['payment_declined','p_value'] < ALPHA)
    or (guardrails.loc['fraud_confirmed','change_pp'] > 0 and guardrails.loc['fraud_confirmed','p_value'] < ALPHA)
)

if primary_positive and its_positive and not adverse_guardrail:
    recommendation = 'Continue / Scale with monitoring'
elif adverse_guardrail:
    recommendation = 'Do not scale yet — investigate guardrail deterioration'
elif primary_positive:
    recommendation = 'Continue with monitoring / validate further'
else:
    recommendation = 'Validate further before scaling'

print('Illustrative recommendation:', recommendation)

## 14. Stakeholder readout standard

A strong readout should answer, in order: What changed? How large was it? Can we trust the data? Was the KPI already trending? Did ramp, seasonality, mix, or campaign affect the comparison? What does the adjusted ITS suggest? Did any guardrail deteriorate? What is the business impact? How strong is causal confidence? What should we do next?

Lead with the decision; place model detail in supporting material.

## Analyst self-review checklist

Before presenting a Pre/Post conclusion, confirm that you defined the intervention and windows, re-derived timing fields, inspected trends, separated ramp from stable Post, evaluated seasonality and population mix, identified concurrent events, reported uncertainty, used a stronger design than a bucketed comparison where possible, inspected rare-event counts, separated association from causality, quantified business impact without false precision, and matched the recommendation to the strength of the evidence.

## Practice questions

1. Why can a statistically significant Pre/Post difference still be misleading?
2. What could make the adjusted ITS result differ materially from the simple Pre/Post result?
3. Why should launch-ramp days be shown rather than silently excluded?
4. Which traffic-mix shifts could influence the primary KPI?
5. Why is fraud percentage change potentially misleading here?
6. Under what conditions would you refuse to make a causal recommendation?
7. How would a matched external comparison group strengthen the design?
8. Which result belongs on slide 1 of the stakeholder readout?
9. What monitoring plan would you recommend after scaling?